# Using Generative LLM APIs in AccFin Research

Generative LLMs are increasingly used in accounting research for textual-analysis tasks that used to require either dictionary/bag-of-words methods or costly manual human coding.

de Kok ([2025](https://doi.org/10.1287/mnsc.2023.03253)) provides a framework for using these models rigorously in accounting research, covering model selection, prompt engineering, and construct validity, and illustrates it with a case study detecting non-answers in earnings conference calls.

### 1. Using OpenAI API

For details, see offical documents at https://github.com/openai/openai-python.

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel

The best practice to handle API keys and passwords is to keep them out of your main Python scripts. The most common approach is to create a `.env` file which contains:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxx
Other_Passwards=xxxxxxxxxxxxxx
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
openai_client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [ ]:
# List available OpenAI models
for model in openai_client.models.list():
    print(model.id)

#### 1.1. Basic Text Prompt

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.5",
    input="Plan a one-day trip to Sydney Australia",
)
print(response.output_text)

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.5",
    instructions="You are a teaching assistant for MBA program.",
    input="Summarize the findings of Sloan (1996) 'Do Stock Prices Fully Reflect Information in Accruals and Cash Flows About Future Earnings?' in one paragraph.",
)
print(response.output_text)

In [ ]:
list_of_CEOs = ['Elon Musk', "Sundar Pichai", "Jeff Bezos", "Mark Zeckerburg", "Jensen Huang"]
birthday_list = []
for ceo in list_of_CEOs:
    response = openai_client.responses.create(
    model="gpt-5.5",
    input=f"""
    What's the birthday of {ceo}, an US entrepreneur?
    Just answer with a date in the format of "YYYYMMDD".
    """,
    )
    birthday_list.append(response.output_text)
print(birthday_list)

#### 1.2. Multi-turn conversations

Pass `previous_response_id` to chain a follow-up onto an earlier response — OpenAI keeps the conversation history server-side, so you don't resend it.

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.5",
    input="Explain what's the *defined contribution plan* in one paragraph.",
)
print(response.output_text)

In [ ]:
# For multi-turn conversations, thread each call to the previous one via previous_response_id
# (the Responses API manages conversation history server-side when store=True, the default)

response = openai_client.responses.create(
    model="gpt-5.5",
    input="What's its difference from *defined benefit plan*?",
    previous_response_id=response.id,
)
print(response.output_text)

Alternatively, you need to build the history yourself by passing `input` as a list of `{"role": ..., "content": ...}` messages:

In [ ]:
history = [
    {"role": "user", "content": "Explain what's the *defined contribution plan* in one paragraph."},
]

response = openai_client.responses.create(model="gpt-5.5", input=history)
print(response.output_text)

# Append the assistant's reply, then the follow-up question
history.append({"role": "assistant", "content": response.output_text})
history.append({"role": "user", "content": "What's its difference from *defined benefit plan*?"})

response = openai_client.responses.create(model="gpt-5.5", input=history)
print(response.output_text)

#### 1.3. Structured JSON output

In [ ]:
class Definition(BaseModel):
    term: str
    definition: str
    confidence: float

response = openai_client.responses.parse(
    model="gpt-5.5",
    input="Define accruals in accounting.",
    text_format=Definition,
)

# Structured Outputs guarantees a schema-conformant result - no manual JSON parsing needed
output = response.output_parsed
output

In [ ]:
# If the rest of your pipeline expects a plain dict rather than a Pydantic object:
output.model_dump()

### 2. Using Google Gemini API

- Get API Key from Google AI Studio: 

    1. Visit [Google AI Studio](https://aistudio.google.com/api-keys) and sign in with a Google account; 

    2. Create a project & get API key: click "Get API Key" (under API Access); save your API key in `.env` file.

- Install the official `google-genai` package.

In [ ]:
from google import genai

In [ ]:
google_client = genai.Client(api_key = os.getenv("GOOGLE_GAI_API"))

In [ ]:
# List available models:
for m in google_client.models.list():
    print(m.name)

#### 2.1. Basic Text Prompt

In [ ]:
interaction = google_client.interactions.create(
    model="gemini-3.5-flash",
    input="Explain what's the *defined contribution plan* in one paragraph.",
)
print(interaction.output_text)

In [ ]:
# For multi-turn conversations, thread each call to the previous one via previous_interaction_id
# (the Interactions API manages conversation history server-side)

interaction = google_client.interactions.create(
    model="gemini-3.5-flash",
    input="What's its difference from *defined benefit plan*?",
    previous_interaction_id=interaction.id,
)
print(interaction.output_text)

#### 2.2 Multimodal Input (e.g., Images)

In [ ]:
import base64

with open("../data/BHP_images/p004_355.jpeg", "rb") as f:
    image_bytes = f.read()

interaction = google_client.interactions.create(
    model="gemini-3.5-flash",
    input=[
        {"type": "text", "text": "Describe what you see."},
        {"type": "image", "mime_type": "image/jpeg", "data": base64.b64encode(image_bytes).decode("utf-8")},
    ],
)
print(interaction.output_text)

Resources:
* [Google Gen AI SDK for Python](https://github.com/googleapis/python-genai)
* [Gemini API Reference](https://ai.google.dev/api)
* [Interactions API guide](https://ai.google.dev/gemini-api/docs/interactions)
* [AI Studio](https://aistudio.google.com/)